<!-- nav -->
[← 4 · Names](04_Names.ipynb) · [Home](../../README.md) · [6 · Calls, members and subscripts →](06_Calls_Members_And_Subscripts.ipynb)

# 5 · Operators

`a + b * c` is not a string of seven characters to a dialect: it is a tree, an addition whose right operand is a
multiplication. This tutorial builds that tree in each language's dialect, in the builders' own calls, and checks that
each dialect's `Text.ToText` writes the language's own text back. Then it shows what the tree decides and the text
does not: where the parentheses go, which operator binds tighter in which language, how a chain of comparisons is held,
which operations a dialect defines, and what each language's arithmetic gives.

The store is the toolkit's `dialects()`, every dialect installed and defined. `S` is its holders' namespace,
`store.Expressions`, and `amd64` the System V ABI's dialect, which C's terms are read under (tutorial 2). `variable`
writes a name, as tutorial 4 showed, wherever a dialect's term is written:

In [1]:
import assert from "node:assert/strict";
import * as C from "@mbse/expressions/C/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import { Domains, Terms } from "@mbse/expressions/Framework";
import { OverflowError } from "@mbse/expressions/Framework/Errors";
import * as Latex from "@mbse/expressions/LaTeX/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as Translators from "@mbse/expressions/Translators";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import { dialects, show } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const Term = S.Framework.Term;
const amd64 = store.Instances.Expressions.SystemV.AMD64;
const variable = (name: string) => (d: any) => d.variable((v: any) => v.name(name));

<a id="T05.1"></a>
## 05.1 · a + b * c is a tree

Patterns, the core vocabulary, has no operators: every operation is a `call` of an operation it names, `callable`, on
its `arguments`, each a term of the framework's union, here a Patterns term. `a + b * c` is `add(a, mul(b, c))`:

In [2]:
const patterns = store.transaction(() => S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
  .callable("add").arguments([
    (t: any) => t.patterns(variable("a")),
    (t: any) => t.patterns((d: any) => d.call((m: any) => m.callable("mul").arguments([
      (t: any) => t.patterns(variable("b")), (t: any) => t.patterns(variable("c"))])))])))));
for (const term of Terms.walk(patterns)) {
  const form = Terms.form(term);
  console.log(form.kind.padEnd(8), form.attributes.get("callable") ?? form.attributes.get("name"), form.arguments.length);
}

call     add 2


variable a 0


call     mul 2


variable b 0


variable c 0


`Terms.walk` visits the tree parents first, each term before its arguments, in order; `Terms.form` reads each one's
kind, its attributes and its arguments. `Terms.fold` goes the other way, children first: it calls a function once per
term with what it gave for that term's arguments, so a tree's depth and its text are each one short function:

In [3]:
function text(term: any, args: string[]): string {
  const form = Terms.form(term);
  return form.kind === "variable" ? `${form.attributes.get("name")}` : `${form.attributes.get("callable")}(${args.join(", ")})`;
}

console.log(Terms.fold(patterns, (_term: any, depths: number[]) => 1 + Math.max(0, ...depths)));
console.log(Terms.fold(patterns, text));
assert.equal(Terms.fold(patterns, text), show(patterns));

3


add(a, mul(b, c))


The toolkit's `show` is that fold, for every kind Patterns has; tutorial 13 writes analyses over constraints this way.

<a id="T05.2"></a>
## 05.2 · The same tree in each language

Each language's dialect has operators, and holds the tree as that language's own grammar does: Python3's `binop`
(Python's `ast.BinOp`, its operator `op`), TypeScript5's and C's `binaryexpression` (`left`, `operator`, `right`),
SystemVerilog's `binaryexpression` too, its names `nameexpression`s of an identifier, Excel's `infix`, MATLAB's and
LaTeX's `binary`. Each is built here in its builders' calls, and each prints the language's own text:

In [4]:
const trees: Map<string, [any, any]> = store.transaction(() => new Map<string, [any, any]>([
  ["Python3", [Python3.Text, S.Python.Python3.Expression((e: any) => e.root((u: any) => u.binop((b: any) => b
    .left(variable("a")).op("+")
    .right((r: any) => r.binop((m: any) => m.left(variable("b")).op("*").right(variable("c")))))))]],
  ["TypeScript5", [TypeScript5.Text, S.TypeScript.TypeScript5.Expression((e: any) => e.root(
    (u: any) => u.binaryexpression((b: any) => b
      .left((l: any) => l.identifier((i: any) => i.name("a"))).operator("+")
      .right((r: any) => r.binaryexpression((m: any) => m
        .left((l: any) => l.identifier((i: any) => i.name("b"))).operator("*")
        .right((r: any) => r.identifier((i: any) => i.name("c"))))))))]],
  ["C", [C.Text, S.C.Expressions.Expression((e: any) => e.root((u: any) => u.binaryexpression((b: any) => b
    .left(variable("a")).operator("+")
    .right((r: any) => r.binaryexpression((m: any) => m.left(variable("b")).operator("*").right(variable("c"))))))
    .abi([amd64]))]],
  ["SystemVerilog", [SystemVerilog.Text, S.SystemVerilog.Expressions.Expression((e: any) => e.root(
    (u: any) => u.binaryexpression((b: any) => b
      .left((l: any) => l.nameexpression((n: any) => n.name((i: any) => i.identifier((d: any) => d.spelling("a")))))
      .operator("+")
      .right((r: any) => r.binaryexpression((m: any) => m
        .left((l: any) => l.nameexpression((n: any) => n.name((i: any) => i.identifier((d: any) => d.spelling("b")))))
        .operator("*")
        .right((r: any) => r.nameexpression((n: any) => n.name((i: any) => i.identifier((d: any) => d.spelling("c"))))))))))]],
  ["Excel", [Excel.Text, S.Excel.Expressions.Expression((e: any) => e.root((u: any) => u.infix((b: any) => b
    .left(variable("a")).operator("+")
    .right((r: any) => r.infix((m: any) => m.left(variable("b")).operator("*").right(variable("c")))))))]],
  ["MATLAB", [Matlab.Text, S.MATLAB.Expressions.Expression((e: any) => e.root((u: any) => u.binary((b: any) => b
    .left(variable("a")).operator("+")
    .right((r: any) => r.binary((m: any) => m.left(variable("b")).operator(".*").right(variable("c")))))))]],
  ["LaTeX", [Latex.Text, S.LaTeX.Expressions.Expression((e: any) => e.root((u: any) => u.binary((b: any) => b
    .left(variable("a")).operator("+")
    .right((r: any) => r.binary((m: any) => m.left(variable("b")).operator("\\cdot").right(variable("c")))))))]],
]));
const native = new Map([["Python3", "a + b * c"], ["TypeScript5", "a + b * c"], ["C", "a + b * c"],
  ["SystemVerilog", "a + b * c"], ["Excel", "=a + b * c"], ["MATLAB", "a + b .* c"], ["LaTeX", "a + b \\cdot c"]]);
for (const [language, [text, tree]] of trees) {
  const written: string = text.ToText(tree);
  console.log(`${language.padEnd(13)} ${written.padEnd(15)} ${Terms.form(Terms.term(tree)).kind}`);
  assert.equal(written, native.get(language));
}

Python3       a + b * c       binop


TypeScript5   a + b * c       binaryexpression


C             a + b * c       binaryexpression


SystemVerilog a + b * c       binaryexpression


Excel         =a + b * c      infix


MATLAB        a + b .* c      binary


LaTeX         a + b \cdot c   binary


Python's text is also read there: Python's twin of this notebook parses it with `Python3.Text.FromText`, which only
Python has, since it reads with Python's own `ast`, and checks that it gives this same tree (`Terms.same`).

<a id="T05.3"></a>
## 05.3 · Parentheses are the printer's, or written

`(a + b) * c` is `mul(add(a, b), c)`: the addition is the multiplication's left operand. No term in that tree is a
parenthesis; each language's printer writes one where its precedence needs it, here in every language. C's tree, in
its builders' calls, and the others translated from Patterns' (tutorial 4), as tutorial 14 does with constraints:

In [5]:
const [product, c] = store.transaction(() => [
  S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c.callable("mul").arguments([
    (t: any) => t.patterns((d: any) => d.call((s: any) => s.callable("add").arguments([
      (t: any) => t.patterns(variable("a")), (t: any) => t.patterns(variable("b"))]))),
    (t: any) => t.patterns(variable("c"))])))),
  S.C.Expressions.Expression((e: any) => e.root((u: any) => u.binaryexpression((b: any) => b
    .left((l: any) => l.binaryexpression((s: any) => s.left(variable("a")).operator("+").right(variable("b"))))
    .operator("*").right(variable("c")))).abi([amd64]))]);
assert.equal(C.Text.ToText(c), "(a + b) * c");
for (const module of [Python3, C, SystemVerilog, Excel, Matlab, Latex] as any[]) {
  const tree = store.transaction(() =>
    Term(Translators.between(Patterns.Expressions.DIALECT, module.Expressions.DIALECT).forward(product)));
  const kinds = new Set([...Terms.walk(tree)].map((term: any) => Terms.form(term).kind));
  console.log(`${module.Expressions.DIALECT.name().padEnd(13)} ${module.Text.ToText(tree).padEnd(17)} ${[...kinds].sort().join(", ")}`);
}

Python3       (a + b) * c       binop, variable


C             (a + b) * c       binaryexpression, variable


SystemVerilog (a + b) * c       binaryexpression, nameexpression


Excel         =(a + b) * c      infix, variable


MATLAB        (a + b) .* c      binary, variable


LaTeX         (a + b) \cdot c   binary, variable


Reading text, the parentheses go: Python's `ast` keeps none, so Python's `Python3.Text.FromText` reads `a + (b * c)`
as the tree of `a + b * c`, and `(a)` as the name `a`, as Python's twin of this notebook shows. What the text meant is
kept; how it was spelled is not.

Parentheses a writer wants kept are data: the dialects whose grammars hold them have a kind for them, Python3's
`parenthesized` (`value`), and TypeScript5's, C's and SystemVerilog's `parenthesizedexpression` (`expression`). Each is
printed as written, needed or not, and is not the same tree as the one without it:

In [6]:
const kept: Map<string, [any, any]> = store.transaction(() => new Map<string, [any, any]>([
  ["Python3", [Python3.Text, S.Python.Python3.Expression((e: any) => e.root((u: any) => u.binop((b: any) => b
    .left(variable("a")).op("+")
    .right((r: any) => r.parenthesized((p: any) => p.value(
      (v: any) => v.binop((m: any) => m.left(variable("b")).op("*").right(variable("c")))))))))]],
  ["TypeScript5", [TypeScript5.Text, S.TypeScript.TypeScript5.Expression((e: any) => e.root(
    (u: any) => u.binaryexpression((b: any) => b
      .left((l: any) => l.identifier((i: any) => i.name("a"))).operator("+")
      .right((r: any) => r.parenthesizedexpression((p: any) => p.expression(
        (x: any) => x.binaryexpression((m: any) => m
          .left((l: any) => l.identifier((i: any) => i.name("b"))).operator("*")
          .right((r: any) => r.identifier((i: any) => i.name("c"))))))))))]],
  ["C", [C.Text, S.C.Expressions.Expression((e: any) => e.root((u: any) => u.binaryexpression((b: any) => b
    .left(variable("a")).operator("+")
    .right((r: any) => r.parenthesizedexpression((p: any) => p.expression(
      (x: any) => x.binaryexpression((m: any) => m.left(variable("b")).operator("*").right(variable("c"))))))))
    .abi([amd64]))]],
]));
for (const [language, [text, tree]] of kept) {
  console.log(`${language.padEnd(13)} ${text.ToText(tree).padEnd(13)} ${Terms.same(tree, trees.get(language)![1])}`);
  assert.equal(text.ToText(tree), "a + (b * c)");
}

Python3       a + (b * c)   false


TypeScript5   a + (b * c)   false


C             a + (b * c)   false


<a id="T05.4"></a>
## 05.4 · Precedence differs

Which operator binds tighter is each language's own, so one tree is written differently in each. A status test,
`eq(bitand(s, 4), 0)`: in Python `&` binds tighter than `==`, and no parenthesis is needed; in C and SystemVerilog
`==` binds tighter than `&`, and the printer writes one; Excel has no `&` operator for bits, but a function, `BITAND`:

In [7]:
const [status, window] = store.transaction(() => [
  S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c.callable("eq").arguments([
    (t: any) => t.patterns((d: any) => d.call((b: any) => b.callable("bitand").arguments([
      (t: any) => t.patterns(variable("s")),
      (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar(4n))))]))),
    (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar(0n))))])))),
  S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c.callable("and").arguments([
    (t: any) => t.patterns((d: any) => d.call((n: any) => n.callable("not").arguments([
      (t: any) => t.patterns((d: any) => d.call((g: any) => g.callable("gt").arguments([
        (t: any) => t.patterns(variable("x")),
        (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar(1n))))])))]))),
    (t: any) => t.patterns((d: any) => d.call((l: any) => l.callable("lt").arguments([
      (t: any) => t.patterns(variable("x")),
      (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar(5n))))])))]))))]);
console.log(show(status), " ", show(window));
for (const module of [Python3, C, SystemVerilog, Excel] as any[]) {
  const written = [status, window].map((tree) => store.transaction(() => module.Text.ToText(Term(Translators.between(
    Patterns.Expressions.DIALECT, module.Expressions.DIALECT).forward(tree)))));
  console.log(`${module.Expressions.DIALECT.name().padEnd(13)} ${written[0].padEnd(17)} ${written[1]}`);
}

eq(bitand(s, 4), 0)   and(not(gt(x, 1)), lt(x, 5))


Python3       s & 4 == 0        not x > 1 and x < 5


C             (s & 4) == 0      !(x > 1) && x < 5


SystemVerilog (s & 4) == 0      !(x > 1) && x < 5


Excel         =BITAND(s, 4) = 0 =AND(NOT(x > 1), x < 5)


`not` is the other way round: Python's `not` binds looser than `>`, so `not x > 1` is `not (x > 1)`, where C's `!`
binds tighter than any binary operator, and the printer writes `!(x > 1)`. A text copied from one language to the other
would change its meaning; the tree, translated, does not. (Python's twin also reads `s & 4 == 0` and translates it
back to this same tree.)

<a id="T05.5"></a>
## 05.5 · Chains and n-ary logic

Some languages' operators take more than two operands. Python's `2.0 <= x <= 8.0` is one `compare`: a `left` operand,
then its `comparisons`, each a `comparison` helper, an `op` and its `comparator`. `a and b and c` is one `boolop` of
three `values`. Each is built in its builders' calls:

In [8]:
const [chain, conjunction] = store.transaction(() => [
  S.Python.Python3.Expression((e: any) => e.root((u: any) => u.compare((c: any) => c
    .left((l: any) => l.literal((r: any) => r.value((v: any) => v.as_scalar(2.0)).text("2.0")))
    .comparisons([
      (k: any) => k.op("<=").comparator(variable("x")),
      (k: any) => k.op("<=").comparator((t: any) => t.literal((r: any) => r.value((v: any) => v.as_scalar(8.0)).text("8.0")))])))),
  S.Python.Python3.Expression((e: any) => e.root((u: any) => u.boolop((b: any) => b
    .op("and").values([variable("a"), variable("b"), variable("c")]))))]);
for (const tree of [chain, conjunction]) {
  const form = Terms.form(Terms.term(tree));
  console.log(`${Python3.Text.ToText(tree).padEnd(16)} ${form.kind.padEnd(8)} ${form.arguments.length} operands`);
}
assert.deepEqual([Python3.Text.ToText(chain), Python3.Text.ToText(conjunction)], ["2.0 <= x <= 8.0", "a and b and c"]);

2.0 <= x <= 8.0  compare  3 operands


a and b and c    boolop   3 operands


Patterns has neither: its `le` and `and` take two arguments. Translated to Patterns, the chain is the `and` of its
links, its middle operand in both, and the `boolop` binary `and`s, the first nested, as Python evaluates them:

In [9]:
for (const tree of [chain, conjunction]) {
  console.log(show(store.transaction(() =>
    Term(Translators.between(Python3.Expressions.DIALECT, Patterns.Expressions.DIALECT).forward(tree)))));
}

and(le(2.0, x), le(x, 8.0))


and(and(a, b), c)


Python's twin also reads both texts, with `FromText`, to these same trees.

<a id="T05.6"></a>
## 05.6 · The operations a dialect defines are data

What Patterns' `callable` may name is not in its code but in the store: Patterns' dialect is a singleton, found by its
global name, which names its **operations**, written by its `define`. `Domains.operations` reads them by name, in the
order the dialect defines them:

In [10]:
const dialect = store.Instances.Expressions.Patterns.Expressions; // the dialect, by its global name
const operations: Map<string, any> = Domains.operations(dialect);
console.log([...operations].map(([name, operation]) => `${name}/${Domains.arity(operation)}`).join(", "));
const add = operations.get("add"); // an operation, by its name
console.log(`${add.name}`, [...add.pattern.variables.keys()].join(", "), `${add.result.name}`);

get/2, has/2, eq/2, ne/2, lt/2, le/2, gt/2, ge/2, and/2, or/2, not/1, implies/2, add/2, sub/2, mul/2, neg/1, bitand/2, bitor/2, bitxor/2, bitnot/1, shl/2, shr/2, count/1, item/2, in/2, sum/1, min/1, max/1, unique/1, entries/2


Expressions.Patterns.Expressions.add a, b Expressions.Patterns.Expressions.Number


Each operation is an expression schema, `Expressions.Patterns.Expressions.add`: its pattern's variables are its
arguments, in order, so `Domains.arity` is their number, and its `result` is what it returns, here a number. They are the core vocabulary every binding simplifies. A call of any other operation is refused by validation, which reads them,
not code: here `div`, which Patterns does not define (division has no single meaning across languages: 05.7), while
05.1's tree, of `add` and `mul`, passes:

In [11]:
const quotient = store.transaction(() => S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
  .callable("div").arguments([(t: any) => t.patterns(variable("a")), (t: any) => t.patterns(variable("b"))])))));
console.log(Patterns.Expressions.DIALECT.validate(quotient, ["a", "b"]));
console.log(Patterns.Expressions.DIALECT.validate(patterns, ["a", "b", "c"]));

[ "'div' is not an operation Patterns defines" ]


[]


The languages' dialects name no operations: their operators are their grammars' kinds, `binop` and the rest, each
operator a string their printers and folding rules know, as 05.2 showed:

In [12]:
console.log(Domains.operations(store.Instances.Expressions.Python.Python3).size,
  Domains.operations(store.Instances.Expressions.C.Expressions).size);

0 0


<a id="T05.7"></a>
## 05.7 · Each language's arithmetic

The same tree can give different values in different languages, because each language's `+` and `/` act on its own
domains (tutorial 2). `2147483647 + 1`, in Patterns, Python and C, each built in its builders' calls, the C one read
under the System V ABI:

In [13]:
const total: Map<string, any> = store.transaction(() => new Map<string, any>([
  ["Patterns", S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c.callable("add").arguments([
    (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar(2147483647n)))),
    (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((v: any) => v.as_scalar(1n))))]))))],
  ["Python3", S.Python.Python3.Expression((e: any) => e.root((u: any) => u.binop((b: any) => b
    .left((l: any) => l.literal((r: any) => r.value((v: any) => v.as_scalar(2147483647n)))).op("+")
    .right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(1n)))))))],
  ["C", S.C.Expressions.Expression((e: any) => e.root((u: any) => u.binaryexpression((b: any) => b
    .left((l: any) => l.literal((r: any) => r.value((v: any) => v.as_scalar(2147483647n)))).operator("+")
    .right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(1n)))))).abi([amd64]))],
]));
assert.equal(Python3.Text.ToText(total.get("Python3")), "2147483647 + 1");
assert.equal(C.Text.ToText(total.get("C")), "2147483647 + 1");
console.log(Patterns.Simplifiers.OfAny(total.get("Patterns")).evaluate(), Python3.Simplifiers.OfAny(total.get("Python3")).evaluate());
try {
  C.Simplifiers.OfAny(total.get("C")).evaluate();
} catch (error) {
  if (!(error instanceof OverflowError)) throw error;
  console.log(error.name, error.message);
}

2147483648n 2147483648n


OverflowError signed overflow in +: 2147483648 does not fit 'int'


Patterns' and Python's integers are ISO/IEC 11404's `integer`, unbounded; C's `int` is the ABI's, 32 bits, and its rules raise where C's behavior is undefined, rather than wrap. The domains say so, read from the store:

In [14]:
for (const [language, holder] of [["Python3", store.Instances.Expressions.Python.Python3], ["C", amd64]] as [string, any][]) {
  const domain = Domains.domains(holder).get("int")!;
  console.log(language.padEnd(8), Domains.kindOf(domain)[1], Domains.properties(domain));
}
console.log(C.Expressions.DIALECT.infer(total.get("C")).name());

Python3  integer Map(3) { "signed" => true, "overflow" => "raise", "ordered" => true }


C        integer Map(6) {
  "width" => 32n,
  "signed" => true,
  "overflow" => "raise",
  "least" => -2147483648n,
  "greatest" => 2147483647n,
  "ordered" => true
}


int


Division differs too. Python's `//` rounds toward negative infinity, C's `/` on integers toward zero; each is its own
operator, in its own dialect, and Patterns defines neither (05.6):

In [15]:
const [floor, truncated] = store.transaction(() => [
  S.Python.Python3.Expression((e: any) => e.root((u: any) => u.binop((b: any) => b
    .left((l: any) => l.unaryop((n: any) => n.op("-").operand((o: any) => o.literal((r: any) => r.value((v: any) => v.as_scalar(7n))))))
    .op("//").right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(2n))))))),
  S.C.Expressions.Expression((e: any) => e.root((u: any) => u.binaryexpression((b: any) => b
    .left((l: any) => l.unaryexpression((n: any) => n.operator("-").operand(
      (o: any) => o.literal((r: any) => r.value((v: any) => v.as_scalar(7n))))))
    .operator("/").right((r: any) => r.literal((r: any) => r.value((v: any) => v.as_scalar(2n)))))).abi([amd64]))]);
console.log(Python3.Text.ToText(floor).padEnd(8), Python3.Simplifiers.OfAny(floor).evaluate());
console.log(C.Text.ToText(truncated).padEnd(8), C.Simplifiers.OfAny(truncated).evaluate());

-7 // 2  -4n


-7 / 2   -3n


Python's twin also reads its two texts, which give the same values.

## Why it works this way

- **The tree is the meaning, the text is one writing of it.** Precedence, parentheses and chains are each language's
  way of writing a tree; tools read and rewrite trees, and only printers and parsers deal in text
  ([PRI-006: text is one more dialect](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-006)).
- **Each language's operators, as its grammar has them.** A dialect holds `binop` or `binaryexpression` because Python's
  `ast` and C's grammar do, so a model is the source's alternative representation, not a paraphrase
  ([CAP-226: complete expression dialects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-226.md)).
- **A neutral vocabulary is data.** Patterns' operations are stored, so a program learns what the core vocabulary can
  say without its code, and validation refuses what it cannot.

---

<!-- nav -->
[← 4 · Names](04_Names.ipynb) · [Home](../../README.md) · [6 · Calls, members and subscripts →](06_Calls_Members_And_Subscripts.ipynb)

<!-- nav -->
[← 4 · Names](04_Names.ipynb) · [Home](../../README.md) · [6 · Calls, members and subscripts →](06_Calls_Members_And_Subscripts.ipynb)